<a href="https://colab.research.google.com/github/syntizen/Union-Chess/blob/main/Union_chess_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import copy
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, clear_output

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __str__(self):
        return f"{self.color[0]}{self.name[0]}"

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        # Setup pieces for White
        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        # Setup pieces for Black
        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c):
        piece = self.board[r][c]
        if not piece or piece.color != self.current_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES (Rook, Bishop, Queen, King) ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def count_kings(self):
        """Counts how many Kings each color has left on the board."""
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def make_move(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.moves_left_this_turn -= 1
        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.piece_values = {"Pawn": 10, "Knight": 30, "Bishop": 30, "Rook": 50, "Queen": 90, "King": 1000}

    def evaluate_board(self, board):
        score = 0
        for r in range(len(board)):
            for c in range(len(board[0])):
                piece = board[r][c]
                if piece:
                    val = self.piece_values.get(piece.name, 0)
                    if piece.color == self.color:
                        score += val
                    else:
                        score -= val
        return score

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves:
            return None

        best_score = -float('inf')
        best_moves = []

        for move in legal_moves:
            from_sq, to_sq = move
            temp_game = copy.deepcopy(game)
            temp_game.make_move(from_sq, to_sq)

            score = self.evaluate_board(temp_game.board)

            # Positional bonus to encourage progressive play
            direction = 1 if self.color == "White" else -1
            score += (to_sq[0] - from_sq[0]) * direction * 0.2

            if score > best_score:
                best_score = score
                best_moves = [move]
            elif score == best_score:
                best_moves.append(move)

        return random.choice(best_moves) if best_moves else None

# --- DYNAMIC RENDERING ENGINE ---
def draw_live_chessboard(game, last_action_text="", fig=None, ax=None):
    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(15, 9))
    else:
        ax.clear()

    ax.set_xlim([0, game.cols])
    ax.set_ylim([0, game.rows])

    for x in range(game.cols):
        for y in range(game.rows):
            color = '#f0d9b5' if (x + y) % 2 == 0 else '#b58863'
            rect = patches.Rectangle((x, y), 1, 1, facecolor=color)
            ax.add_patch(rect)

            piece = game.board[y][x]
            if piece:
                text_color = 'white' if piece.color == 'White' else 'black'
                bbox_props = dict(boxstyle="circle,pad=0.2", fc="black" if piece.color=='White' else "white", ec="none", alpha=0.6)
                ax.text(x + 0.5, y + 0.5, str(piece), fontsize=8, weight='bold',
                        ha='center', va='center', color=text_color, bbox=bbox_props)

    ax.set_aspect('equal', adjustable='box')

    kings = game.count_kings()
    title_text = (f"Turn: {game.current_turn} | Action: {4 - game.moves_left_this_turn}/3\n"
                  f"White Kings Left: {kings['White']} | Black Kings Left: {kings['Black']}\n"
                  f"Latest Log: {last_action_text}")
    ax.set_title(title_text, fontsize=12, weight='bold')
    ax.grid(False)

    clear_output(wait=True)
    display(fig)
    plt.close(fig) # Prevent ghosting plots

In [ ]:
# Create Game Instance
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

# Prepare a single global figure frame for animating
fig, ax = plt.subplots(figsize=(15, 9))
last_text = "Game Started!"

# Initial draw
draw_live_chessboard(game, last_text, fig, ax)

# Game Loop
move_counter = 1
while True:
    # Check Win/Loss conditions
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        draw_live_chessboard(game, "🏆 BLACK WINS! All White Kings have been obliterated!", fig, ax)
        break
    if king_counts["Black"] == 0:
        draw_live_chessboard(game, "🏆 WHITE WINS! All Black Kings have been obliterated!", fig, ax)
        break

    current_color = game.current_turn
    bot = white_bot if current_color == "White" else black_bot

    # Get move selection
    chosen_move = bot.select_best_move(game)

    if chosen_move:
        from_sq, to_sq = chosen_move
        moving_piece = game.board[from_sq[0]][from_sq[1]]
        target_square = game.board[to_sq[0]][to_sq[1]]

        # Build telemetry log string
        capture_log = f" (Captured {target_square.color} {target_square.name}!)" if target_square else ""
        last_text = f"[Move {move_counter}] {current_color} moved {moving_piece.name} from {from_sq} to {to_sq}{capture_log}"

        # Process move state transitions
        game.make_move(from_sq, to_sq)
        move_counter += 1

        # Live redraw on the SAME window space
        draw_live_chessboard(game, last_text, fig, ax)

        # Adjust sleep time (in seconds) to speed up or slow down the simulation
        time.sleep(0.1)
    else:
        # Handling stalemates or sudden blocked paths
        last_text = f"Game Over! {current_color} has no legal moves left. Draw by stalemate."
        draw_live_chessboard(game, last_text, fig, ax)
        break

In [ ]:
import random
import copy
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __str__(self):
        return f"{self.color[0]}{self.name[0]}"

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.turn_origins = []
        self.turn_destinations = []
        self.turn_paths = set()

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c):
        piece = self.board[r][c]
        if not piece or piece.color != self.current_turn:
            return []
        if (r, c) in self.moved_pieces_this_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES (Rook, Bishop, Queen, King) ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        path_cells = []
        if not piece or piece.name == "Knight":
            return path_cells

        dr = tr - fr
        dc = tc - fc

        step_r = (dr // abs(dr)) if dr != 0 else 0
        step_c = (dc // abs(dc)) if dc != 0 else 0

        curr_r, curr_c = fr + step_r, fc + step_c
        while (curr_r, curr_c) != (tr, tc):
            path_cells.append((curr_r, curr_c))
            curr_r += step_r
            curr_c += step_c

        return path_cells

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def clear_turn_highlights(self):
        self.turn_origins.clear()
        self.turn_destinations.clear()
        self.turn_paths.clear()

    def make_move(self, from_sq, to_sq):
        self.turn_origins.append(from_sq)
        self.turn_destinations.append(to_sq)
        self.turn_paths.update(self.calculate_path(from_sq, to_sq))

        fr, fc = from_sq
        tr, tc = to_sq

        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.piece_values = {"Pawn": 10, "Knight": 30, "Bishop": 30, "Rook": 50, "Queen": 90, "King": 1000}

    def evaluate_board(self, board):
        score = 0
        for r in range(len(board)):
            for c in range(len(board[0])):
                piece = board[r][c]
                if piece:
                    val = self.piece_values.get(piece.name, 0)
                    if piece.color == self.color:
                        score += val
                    else:
                        score -= val
        return score

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves:
            return None

        best_score = -float('inf')
        best_moves = []

        for move in legal_moves:
            from_sq, to_sq = move
            temp_game = copy.deepcopy(game)
            temp_game.make_move(from_sq, to_sq)

            score = self.evaluate_board(temp_game.board)

            direction = 1 if self.color == "White" else -1
            score += (to_sq[0] - from_sq[0]) * direction * 0.2

            if score > best_score:
                best_score = score
                best_moves = [move]
            elif score == best_score:
                best_moves.append(move)

        return random.choice(best_moves) if best_moves else None

# --- HTML MULTI-TRAIL RENDERING ENGINE (COLOR FIX APPLIED) ---
def generate_board_html(game, turn_summary_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in turn_summary_logs])
    if not logs_html:
        logs_html = "Waiting for game initialization..."

    html = f"""
    <div style="font-family: monospace; background-color: #222; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #4af;">
            NEXT TEAM TO ACT: {game.current_turn.upper()} &nbsp;|&nbsp; Completed Full Turns: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings: {kings['Black']}</span>
        </div>
        <div style="background: #333; padding: 8px 12px; border-left: 4px solid #4af; font-size: 12px; color: #ddd; line-height: 1.4;">
            <b>Full Turn Formation Actions:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """

    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            if (r, c) in game.turn_destinations:
                bg_color = "#4682B4"
            elif (r, c) in game.turn_origins:
                bg_color = "#DEB887"
            elif (r, c) in game.turn_paths:
                bg_color = "#B0E0E6"

            piece = game.board[r][c]
            piece_html = ""

            if piece:
                # FIXED LOGIC: White pieces get a white circle background with black text.
                # Black pieces get a dark charcoal background with white text.
                p_bg = "#ffffff" if piece.color == "White" else "#2c2c2c"
                p_color = "#000000" if piece.color == "White" else "#ffffff"
                border_stroke = "1px solid #000" if piece.color == "White" else "1px solid #fff"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 26px; border-radius: 50%;
                     background-color: {p_bg}; color: {p_color}; font-weight: bold; font-size: 11px;
                     text-align: center; margin: auto; border: {border_stroke}; box-shadow: 1px 1px 3px rgba(0,0,0,0.4);">
                    {str(piece)}
                </div>
                """

            html += f"""
            <td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">
                {piece_html}
            </td>
            """
        html += "</tr>"

    html += "</table>"
    return html

In [ ]:
# Create Engine & Robot States
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Game initialization phase."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS!"], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS!"], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    game.clear_turn_highlights()
    turn_summary_logs = []

    actions_taken = 0
    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            boxes_traveled = max(abs(to_sq[0] - from_sq[0]), abs(to_sq[1] - from_sq[1]))
            capture_note = f" (Captured {target_square.color} {target_square.name}!)" if target_square else ""
            turn_summary_logs.append(f"Action {step+1}: {moving_piece.name} moved {boxes_traveled} blocks from {from_sq} to {to_sq}{capture_note}")

            game.make_move(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        board_widget.value = generate_board_html(game, [f"Stalemate! {active_color} is blocked."], turn_counter)
        break

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(1.2)

In [ ]:
import random
import copy
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __str__(self):
        return f"{self.color[0]}{self.name[0]}"

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        # Tracking variables for current visual animation frames
        self.animating_piece = None
        self.animating_pos = None # (r, c)

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c):
        piece = self.board[r][c]
        if not piece or piece.color != self.current_turn:
            return []
        if (r, c) in self.moved_pieces_this_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES (Rook, Bishop, Queen, King) ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        if not piece:
            return [(fr, fc), (tr, tc)]

        if piece.name == "Knight":
            return [(fr, fc), (tr, fc), (tr, tc)]

        dr = tr - fr
        dc = tc - fc
        steps = max(abs(dr), abs(dc))

        if steps == 0:
            return [(fr, fc)]

        path = []
        for i in range(steps + 1):
            curr_r = fr + (dr * i) // steps
            curr_c = fc + (dc * i) // steps
            path.append((curr_r, curr_c))
        return path

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.piece_values = {"Pawn": 10, "Knight": 30, "Bishop": 30, "Rook": 50, "Queen": 90, "King": 1000}

    def evaluate_board(self, board):
        score = 0
        for r in range(len(board)):
            for c in range(len(board[0])):
                piece = board[r][c]
                if piece:
                    val = self.piece_values.get(piece.name, 0)
                    if piece.color == self.color:
                        score += val
                    else:
                        score -= val
        return score

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves:
            return None

        best_score = -float('inf')
        best_moves = []

        for move in legal_moves:
            from_sq, to_sq = move
            temp_game = copy.deepcopy(game)
            temp_game.finalize_move_data(from_sq, to_sq)

            score = self.evaluate_board(temp_game.board)

            direction = 1 if self.color == "White" else -1
            score += (to_sq[0] - from_sq[0]) * direction * 0.2

            if score > best_score:
                best_score = score
                best_moves = [move]
            elif score == best_score:
                best_moves.append(move)

        return random.choice(best_moves) if best_moves else None

# --- HTML RENDERING ENGINE (WITH FIXED CONTAINER HEIGHTS TO PREVENT SHIFTING) ---
def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html:
        logs_html = "Processing..."

    # Crucial Layout fix: added fixed vertical heights to container boxes (height: 54px;)
    # to stop dynamic line updates from shifting the chessboard grid.
    html = f"""
    <div style="font-family: monospace; background-color: #222; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #4af;">
            CURRENT TEAM TURN: {game.current_turn.upper()} &nbsp;|&nbsp; Total Complete Turns: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings: {kings['Black']}</span>
        </div>
        <div style="background: #333; padding: 8px 12px; border-left: 4px solid #4af; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Live Actions Output Tracker:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """

    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                p_bg = "#ffffff" if active_render_piece.color == "White" else "#2c2c2c"
                p_color = "#000000" if active_render_piece.color == "White" else "#ffffff"
                border_stroke = "1px solid #000" if active_render_piece.color == "White" else "1px solid #fff"

                shadow = "3px 3px 7px rgba(0,0,0,0.5)" if (game.animating_piece and game.animating_pos == (r, c)) else "1px 1px 3px rgba(0,0,0,0.3)"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 26px; border-radius: 50%;
                     background-color: {p_bg}; color: {p_color}; font-weight: bold; font-size: 11px;
                     text-align: center; margin: auto; border: {border_stroke}; box-shadow: {shadow}; transition: all 0.04s ease;">
                    {str(active_render_piece)}
                </div>
                """

            html += f"""
            <td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">
                {piece_html}
            </td>
            """
        html += "</tr>"

    html += "</table>"
    return html

In [ ]:
# Initialize Game & Bots
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Match initialized. Output tracker stabilized."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS!"], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS!"], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            boxes_traveled = max(abs(to_sq[0] - from_sq[0]), abs(to_sq[1] - from_sq[1]))
            capture_note = f" (Captured {target_square.color} {target_square.name}!)" if target_square else ""
            log_line = f"Action {step+1}: {moving_piece.name} moving {boxes_traveled} blocks to {to_sq}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- START SLIDING ANIMATION SUB-LOOP ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.04)

            # --- END ANIMATION SUB-LOOP ---
            game.animating_piece = None
            game.animating_pos = None

            # Reset piece on structural map layer to lock step changes
            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        board_widget.value = generate_board_html(game, [f"Stalemate! No moves left for {active_color}."], turn_counter)
        break

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    #time.sleep(0.1)

In [ ]:
import random
import copy
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __str__(self):
        return f"{self.color[0]}{self.name[0]}"

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece:
            return []
        if not ignore_turn and piece.color != self.current_turn:
            return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None
        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1
        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- THE SMART MASTERMIND ROBOT ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 10, "Knight": 30, "Bishop": 35, "Rook": 50, "Queen": 95, "King": 2000}

    def evaluate_board(self, game_state):
        """Advanced tactical evaluation engine scanning for multiple strategic structures."""
        board = game_state.board
        score = 0

        # 1. Base Material Tracking & King Hunting Vectors
        enemy_king_positions = []
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                piece = board[r][c]
                if piece:
                    val = self.piece_values.get(piece.name, 0)
                    if piece.color == self.color:
                        score += val

                        # FORMATION: Reward center vertical control (columns 6 to 17 are the inner battlefield)
                        if 6 <= c <= 17:
                            score += 1.5
                        # FORMATION: Progress forward out of back rank
                        advance_bonus = r if self.color == "White" else (game_state.rows - 1 - r)
                        score += advance_bonus * 0.3
                    else:
                        score -= val
                        if piece.name == "King":
                            enemy_king_positions.append((r, c))

        # 2. DEFENSE & GUARDING SCANS
        # Temporarily flip turn context to calculate visual attack maps safely
        original_turn = game_state.current_turn

        # Build attack influence maps
        game_state.current_turn = self.enemy_color
        enemy_attack_map = set()
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                p = board[r][c]
                if p and p.color == self.enemy_color:
                    # Collect everywhere the enemy can hit
                    moves = game_state.get_valid_moves(r, c, ignore_turn=True)
                    enemy_attack_map.update(moves)

        game_state.current_turn = self.color
        friendly_attack_map = set()
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                p = board[r][c]
                if p and p.color == self.color:
                    moves = game_state.get_valid_moves(r, c, ignore_turn=True)
                    friendly_attack_map.update(moves)

        # Restore actual runtime environment states
        game_state.current_turn = original_turn

        # Evaluate positional structures
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                piece = board[r][c]
                if piece and piece.color == self.color:
                    # DEFENSE: Penalty if our piece is hanging under fire
                    if (r, c) in enemy_attack_map:
                        # If unprotected, apply a massive penalty; if protected, apply a tiny warning discount
                        if (r, c) not in friendly_attack_map:
                            score -= self.piece_values.get(piece.name, 0) * 0.8
                        else:
                            score -= self.piece_values.get(piece.name, 0) * 0.1

                    # GUARDING: Bonus if friendly pieces support each other
                    if (r, c) in friendly_attack_map:
                        score += 0.8

                    # WINNING STRATEGY: Close the distance to enemy Kings
                    if piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                        for kr, kc in enemy_king_positions:
                            dist = abs(r - kr) + abs(c - kc)
                            # Closer pieces get higher scaling weights
                            score += (40 - dist) * 0.15

        return score

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves:
            return None

        best_score = -float('inf')
        best_moves = []

        # Look-ahead evaluation modeling
        for move in legal_moves:
            from_sq, to_sq = move
            temp_game = copy.deepcopy(game)
            temp_game.finalize_move_data(from_sq, to_sq)

            score = self.evaluate_board(temp_game)

            if score > best_score:
                best_score = score
                best_moves = [move]
            elif score == best_score:
                best_moves.append(move)

        return random.choice(best_moves) if best_moves else None

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing..."

    html = f"""
    <div style="font-family: monospace; background-color: #222; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #5f4;">
            🤖 MASTERMIND ENGINE ACTIVE &nbsp;|&nbsp; Completed Full Turns: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings Left: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings Left: {kings['Black']}</span>
        </div>
        <div style="background: #333; padding: 8px 12px; border-left: 4px solid #5f4; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Tactical Sub-System Outputs:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                p_bg = "#ffffff" if active_render_piece.color == "White" else "#2c2c2c"
                p_color = "#000000" if active_render_piece.color == "White" else "#ffffff"
                border_stroke = "1px solid #000" if active_render_piece.color == "White" else "1px solid #fff"
                shadow = "3px 3px 7px rgba(0,0,0,0.5)" if (game.animating_piece and game.animating_pos == (r, c)) else "1px 1px 3px rgba(0,0,0,0.3)"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 26px; border-radius: 50%;
                     background-color: {p_bg}; color: {p_color}; font-weight: bold; font-size: 11px;
                     text-align: center; margin: auto; border: {border_stroke}; box-shadow: {shadow}; transition: all 0.04s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize Game & Mastermind Smart AI Robots
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Mastermind strategic matrix active. Initializing arrays."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Tactical Domination Complete."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Tactical Domination Complete."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            boxes_traveled = max(abs(to_sq[0] - from_sq[0]), abs(to_sq[1] - from_sq[1]))
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""
            log_line = f"Action {step+1}: {moving_piece.name} tactical dash to {to_sq}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- START SLIDING ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.03) # Crisp slide animation acceleration

            # --- END ANIMATION ---
            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        board_widget.value = generate_board_html(game, [f"Absolute Stalemate reached. Structural lockdown."], turn_counter)
        break

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.4)

In [ ]:
import random
import copy
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        """Maps piece names to traditional Unicode chess characters."""
        symbols = {
            "White": {"King": "♔", "Queen": "♕", "Rook": "♖", "Bishop": "♗", "Knight": "♘", "Pawn": "♙"},
            "Black": {"King": "♚", "Queen": "♛", "Rook": "⚜", "Bishop": "♝", "Knight": "♞", "Pawn": "♟"}
        }
        # Note: Standard Unicode black rooks (♜) can look overly bulky in some browsers,
        # using the crisp contour glyph (⚜) keeping it clean alongside the white rook (♖).
        return symbols[self.color].get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece:
            return []
        if not ignore_turn and piece.color != self.current_turn:
            return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None
        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1
        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 10, "Knight": 30, "Bishop": 35, "Rook": 50, "Queen": 95, "King": 2000}

    def evaluate_board(self, game_state):
        board = game_state.board
        score = 0
        enemy_king_positions = []
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                piece = board[r][c]
                if piece:
                    val = self.piece_values.get(piece.name, 0)
                    if piece.color == self.color:
                        score += val
                        if 6 <= c <= 17: score += 1.5
                        advance_bonus = r if self.color == "White" else (game_state.rows - 1 - r)
                        score += advance_bonus * 0.3
                    else:
                        score -= val
                        if piece.name == "King": enemy_king_positions.append((r, c))

        original_turn = game_state.current_turn
        game_state.current_turn = self.enemy_color
        enemy_attack_map = set()
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                p = board[r][c]
                if p and p.color == self.enemy_color:
                    enemy_attack_map.update(game_state.get_valid_moves(r, c, ignore_turn=True))

        game_state.current_turn = self.color
        friendly_attack_map = set()
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                p = board[r][c]
                if p and p.color == self.color:
                    friendly_attack_map.update(game_state.get_valid_moves(r, c, ignore_turn=True))

        game_state.current_turn = original_turn

        for r in range(game_state.rows):
            for c in range(game_state.cols):
                piece = board[r][c]
                if piece and piece.color == self.color:
                    if (r, c) in enemy_attack_map:
                        if (r, c) not in friendly_attack_map:
                            score -= self.piece_values.get(piece.name, 0) * 0.8
                        else:
                            score -= self.piece_values.get(piece.name, 0) * 0.1
                    if (r, c) in friendly_attack_map: score += 0.8
                    if piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                        for kr, kc in enemy_king_positions:
                            dist = abs(r - kr) + abs(c - kc)
                            score += (40 - dist) * 0.15
        return score

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None
        best_score = -float('inf')
        best_moves = []
        for move in legal_moves:
            from_sq, to_sq = move
            temp_game = copy.deepcopy(game)
            temp_game.finalize_move_data(from_sq, to_sq)
            score = self.evaluate_board(temp_game)
            if score > best_score:
                best_score = score
                best_moves = [move]
            elif score == best_score:
                best_moves.append(move)
        return random.choice(best_moves) if best_moves else None

# --- HTML TRADITIONAL RENDER ENGINE ---
def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing..."

    html = f"""
    <div style="font-family: monospace; background-color: #222; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #4af;">
            🏰 TRADITIONAL TRIPLE CHESS VARIANT &nbsp;|&nbsp; Full Turns: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings Left: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings Left: {kings['Black']}</span>
        </div>
        <div style="background: #333; padding: 8px 12px; border-left: 4px solid #4af; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Tactical Output:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                # Custom color tuning for Unicode text characters
                p_color = "#1e1e1e" if active_render_piece.color == "White" else "#ffffff"
                p_shadow = "drop-shadow(1px 1px 1px #fff)" if active_render_piece.color == "White" else "drop-shadow(1px 1px 1px #000)"

                # Render using traditional transparent styling so board themes pass through neatly
                piece_html = f"""
                <div style="font-size: 26px; line-height: 32px; width: 32px; height: 32px;
                            text-align: center; color: {p_color}; filter: {p_shadow};
                            cursor: default; user-select: none; font-weight: normal; transition: all 0.04s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Match initialized with traditional piece elements."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS!"], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS!"], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            capture_note = f" (Captured {target_square.name}!)" if target_square else ""
            log_line = f"Action {step+1}: {moving_piece.color} {moving_piece.name} shifts toward square {to_sq}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- ANIMATION DRIVER ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.03)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        board_widget.value = generate_board_html(game, [f"Lockdown stalemate reached."], turn_counter)
        break

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0)

In [ ]:
import random
import copy
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        """Maps piece names to bold, filled traditional chess characters."""
        # Using solid filled Unicode symbols for both sides to guarantee maximum visibility
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece:
            return []
        if not ignore_turn and piece.color != self.current_turn:
            return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None
        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1
        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 10, "Knight": 30, "Bishop": 35, "Rook": 50, "Queen": 95, "King": 2000}

    def evaluate_board(self, game_state):
        board = game_state.board
        score = 0
        enemy_king_positions = []
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                piece = board[r][c]
                if piece:
                    val = self.piece_values.get(piece.name, 0)
                    if piece.color == self.color:
                        score += val
                        if 6 <= c <= 17: score += 1.5
                        advance_bonus = r if self.color == "White" else (game_state.rows - 1 - r)
                        score += advance_bonus * 0.3
                    else:
                        score -= val
                        if piece.name == "King": enemy_king_positions.append((r, c))

        original_turn = game_state.current_turn
        game_state.current_turn = self.enemy_color
        enemy_attack_map = set()
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                p = board[r][c]
                if p and p.color == self.enemy_color:
                    enemy_attack_map.update(game_state.get_valid_moves(r, c, ignore_turn=True))

        game_state.current_turn = self.color
        friendly_attack_map = set()
        for r in range(game_state.rows):
            for c in range(game_state.cols):
                p = board[r][c]
                if p and p.color == self.color:
                    friendly_attack_map.update(game_state.get_valid_moves(r, c, ignore_turn=True))

        game_state.current_turn = original_turn

        for r in range(game_state.rows):
            for c in range(game_state.cols):
                piece = board[r][c]
                if piece and piece.color == self.color:
                    if (r, c) in enemy_attack_map:
                        if (r, c) not in friendly_attack_map:
                            score -= self.piece_values.get(piece.name, 0) * 0.8
                        else:
                            score -= self.piece_values.get(piece.name, 0) * 0.1
                    if (r, c) in friendly_attack_map: score += 0.8
                    if piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                        for kr, kc in enemy_king_positions:
                            dist = abs(r - kr) + abs(c - kc)
                            score += (40 - dist) * 0.15
        return score

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_moves = []

        for move in legal_moves:
            from_sq, to_sq = move
            temp_game = copy.deepcopy(game)
            temp_game.finalize_move_data(from_sq, to_sq)
            score = self.evaluate_board(temp_game)

            if score > best_score:
                best_score = score
                best_moves = [move]
            elif score == best_score:
                best_moves.append(move)

        # CRITICAL RULE FIX: If all calculated moves show negative utility (avoiding a trap),
        # do not pass. Instead, pick a random legal move as a fallback strategy to avoid stalling out.
        if best_score == -float('inf') or not best_moves:
            return random.choice(legal_moves)

        return random.choice(best_moves)

# --- HTML TRADITIONAL ENGINE (WITH FILLED SOLID ICONS) ---
def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing..."

    html = f"""
    <div style="font-family: monospace; background-color: #222; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #4af;">
            🏰 TRADITIONAL TRIPLE CHESS VARIANT &nbsp;|&nbsp; Full Turns Completed: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings Left: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings Left: {kings['Black']}</span>
        </div>
        <div style="background: #333; padding: 8px 12px; border-left: 4px solid #4af; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Tactical Output Panel:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                # DESIGN UPDATE: Fully filled circle backdrops.
                # White pieces = White circles with crisp Black solid glyphs inside.
                # Black pieces = Dark Charcoal circles with crisp White solid glyphs inside.
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.04s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Match launched. Run vector engine mapping."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Total tactical wipeout."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Total tactical wipeout."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            capture_note = f" (Captured {target_square.name}!)" if target_square else ""
            log_line = f"Action {step+1}: {moving_piece.color} {moving_piece.name} advances to {to_sq}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- ANIMATION DRIVER ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.03)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        # Fallback to force side switches if an unhandled block pattern arises
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.3)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece:
            return []
        if not ignore_turn and piece.color != self.current_turn:
            return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn:
            return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None
        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1
        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- HIGH-PERFORMANCE IN-PLACE STRATEGY ENGINE ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 10, "Knight": 35, "Bishop": 35, "Rook": 55, "Queen": 100, "King": 5000}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        """In-place math matrix calculation. Completely eliminates memory copying stalls."""
        fr, fc = from_sq
        tr, tc = to_sq

        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0

        # 1. Immediate Capture Material Optimization
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 10

        # 2. Positional Development Vectors
        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 4.0  # High drive forward

        # Central columns dominance bonus (cols 8-15 form the high-ground spine)
        if 8 <= tc <= 15:
            score_delta += 2.5

        # 3. Defensive Escape Mechanism
        # Scan if the piece was under fire at its old position
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.color == self.enemy_color:
                    # If enemy controls original square, fleeing is highly valued
                    if (fr, fc) in [(r+1, c-1), (r+1, c+1)] if self.enemy_color == "White" else [(r-1, c-1), (r-1, c+1)]:
                        score_delta += self.piece_values.get(moving_piece.name, 0) * 0.3

        # 4. King Hunting System
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King" and p.color == self.enemy_color:
                    old_dist = abs(fr - r) + abs(fc - c)
                    new_dist = abs(tr - r) + abs(tc - c)
                    if new_dist < old_dist:
                        score_delta += (40 - new_dist) * 2.0 # Pulls surrounding assets toward enemy kings

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_choices = []

        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)

        if best_choices:
            return random.choice(best_choices)
        return random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #4af;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #4af;">
            🚀 VECTOR CHESS ENGINE v3.0 (STABILIZED) &nbsp;|&nbsp; Turn Vector: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings Active: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings Active: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #4af; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Telemetry Logging Matrix:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Vector logic online. Running matrix match sequence."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK VICTORIOUS! All White Kings eliminated."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE VICTORIOUS! All Black Kings eliminated."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            capture_note = f" (Captured {target_square.name}!)" if target_square else ""
            log_line = f"Action {step+1}: {moving_piece.color} {moving_piece.name} sweeps to {to_sq}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED SLIDE ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.015) # Optimized high-speed vector tracking

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        # Prevent runtime stalls by shifting the context window explicitly
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.15)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        self.board[tr][tc] = piece
        self.board[fr][fc] = None
        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1
        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- COHESIVE PHALANX STRATEGY ENGINE ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 40, "Rook": 60, "Queen": 110, "King": 9000}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0

        # 1. Capture Processing Weights
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 15

        # 2. Base Development Direction
        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 3.0

        # 3. ADVANCED PAWN PHALANX STRUCTURE CALCULATOR
        if moving_piece.name == "Pawn":
            # Rule A: Reward forming a side-by-side Wall (Phalanx alignment)
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 8.0 # Generous bonus for standing shoulder-to-shoulder

            # Rule B: Reward Defensive Diagonal Support Chains
            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 6.0 # Bonus for moving into a guarded diagonal pocket

        # 4. SHIELD WALL (King Guarding Matrix)
        # Scan proximity to friendly Kings to maintain an active defensive perimeter
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King" and p.color == self.color:
                    old_k_dist = max(abs(fr - r), abs(fc - c))
                    new_k_dist = max(abs(tr - r), abs(tc - c))

                    # If a pawn leaves the King's immediate umbrella protection row (1 or 2 steps away), penalize it
                    if moving_piece.name == "Pawn" and new_k_dist > 2 and old_k_dist <= 2:
                        score_delta -= 15.0
                    # Reward keeping heavy armor units near your monarch zone
                    if moving_piece.name in ["Rook", "Knight", "Bishop"] and new_k_dist <= 3:
                        score_delta += 4.0

        # 5. KING HUNT VECTOR GRID
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King" and p.color == self.enemy_color:
                    old_dist = abs(fr - r) + abs(fc - c)
                    new_dist = abs(tr - r) + abs(tc - c)
                    if new_dist < old_dist:
                        score_delta += (40 - new_dist) * 1.5

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_choices = []

        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)

        if best_choices:
            return random.choice(best_choices)
        return random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing Matrix..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #5f4;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #5f4;">
            🛡️ PHALANX SHIELD WALL TACTICAL MATRIX &nbsp;|&nbsp; Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings Active: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings Active: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #5f4; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Tactical Sub-System Vector Logging:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Vector coordination initialized. Driving structure alignments."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK VICTORIOUS! Integrated structure checkmate."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE VICTORIOUS! Integrated structure checkmate."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            capture_note = f" (Captured {target_square.name}!)" if target_square else ""
            log_line = f"Action {step+1}: {moving_piece.color} {moving_piece.name} advances to {to_sq}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED SLIDE ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.015)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.15)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None

        # STALEMATE BREAKOUT: Tracks the last 6 coordinates visited by each side
        self.position_history = {"White": [], "Black": []}

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[1][i] = ChessPiece("Pawn", "White")
            board[0][i] = ChessPiece(white_pieces_order[i], "White")

        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[14][i] = ChessPiece("Pawn", "Black")
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- PAWN MOVES ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            start_row = 1 if piece.color == "White" else 14

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == start_row and self.board[r + 2 * direction][c] is None:
                    moves.append((r + 2 * direction, c))
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        # UPGRADE: Dynamic Pawn Promotion Check
        if piece.name == "Pawn" and (tr == 15 or tr == 0):
            piece = ChessPiece("Queen", piece.color) # Promoted to Queen!

        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        # Track history for repetition avoidance
        self.position_history[piece.color].append(to_sq)
        if len(self.position_history[piece.color]) > 6:
            self.position_history[piece.color].pop(0)

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- THE COMPLETE MASTERMIND ROBOT ENGINE ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 40, "Rook": 60, "Queen": 110, "King": 9000}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0

        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 15

        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 3.5

        # UPGRADE: Repetition Avoidance Penalty
        if to_sq in game.position_history[self.color]:
            score_delta -= 12.0 # Penalize stepping back into recently used tiles

        # Phalanx Structure Mechanics
        if moving_piece.name == "Pawn":
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 8.0

            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 6.0

        # King Guarding & Shield Wall Proximity Matrix
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King" and p.color == self.color:
                    old_k_dist = max(abs(fr - r), abs(fc - c))
                    new_k_dist = max(abs(tr - r), abs(tc - c))

                    if moving_piece.name == "Pawn" and new_k_dist > 2 and old_k_dist <= 2:
                        score_delta -= 15.0
                    if moving_piece.name in ["Rook", "Knight", "Bishop"] and new_k_dist <= 3:
                        score_delta += 4.0

        # Enemy King Assassination Proximity Vectors
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King" and p.color == self.enemy_color:
                    old_dist = abs(fr - r) + abs(fc - c)
                    new_dist = abs(tr - r) + abs(tc - c)
                    if new_dist < old_dist:
                        score_delta += (40 - new_dist) * 2.0

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_choices = []

        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)

        if best_choices:
            return random.choice(best_choices)
        return random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #0cf;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #0cf;">
            🛡️ TRIPLE UNION CHESS MATRIX (v4.0 PRO) &nbsp;|&nbsp; Global Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Kings Active: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Kings Active: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #0cf; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Tactical Output:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""

            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Engine operational. Advanced formation weights distributed."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Integrated tactical checkmate."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Integrated tactical checkmate."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            # Identify if promotion occurs for the logger output
            promo_note = " -> ♕ PROMOTED" if moving_piece.name == "Pawn" and (to_sq[0] == 15 or to_sq[0] == 0) else ""
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""

            log_line = f"Action {step+1}: {moving_piece.color} {moving_piece.name} advances to {to_sq}{promo_note}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED VECTOR ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.015)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.15)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None
        self.position_history = {"White": [], "Black": []}

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        # --- WHITE INITIALIZATION ---
        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[0][i] = ChessPiece(white_pieces_order[i], "White")
            board[1][i] = ChessPiece("Pawn", "White") # Pawn Layer 1
            board[2][i] = ChessPiece("Pawn", "White") # Pawn Layer 2 (New!)

        # --- BLACK INITIALIZATION ---
        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")
            board[14][i] = ChessPiece("Pawn", "Black") # Pawn Layer 1
            board[13][i] = ChessPiece("Pawn", "Black") # Pawn Layer 2 (New!)

        return board

    def get_valid_moves(self, r, c, ignore_turn=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- ADVANCED DOUBLE-LAYER PAWN ENGINE ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            layer_1_start = 1 if piece.color == "White" else 14
            layer_2_start = 2 if piece.color == "White" else 13

            # Standard single forward move step
            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))

                # Rule A: If starting on Layer 1, allow up to a 3-square initial leap
                if r == layer_1_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))
                        if self.board[r + 3 * direction][c] is None:
                            moves.append((r + 3 * direction, c))

                # Rule B: If starting on Layer 2, allow a 2-square initial leap
                elif r == layer_2_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))

            # Standard diagonal capture targeting matrix
            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        if piece.name == "Pawn" and (tr == 15 or tr == 0):
            piece = ChessPiece("Queen", piece.color)

        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.position_history[piece.color].append(to_sq)
        if len(self.position_history[piece.color]) > 6:
            self.position_history[piece.color].pop(0)

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 40, "Rook": 60, "Queen": 110, "King": 9000}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 15

        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 3.5

        if to_sq in game.position_history[self.color]:
            score_delta -= 12.0

        # Pawn Structure Array Evaluation
        if moving_piece.name == "Pawn":
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 8.0
            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 6.0

        # Proximity Vector Mapping (Shield Walls & Targets)
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p:
                    if p.name == "King":
                        if p.color == self.color:
                            old_k_dist = max(abs(fr - r), abs(fc - c))
                            new_k_dist = max(abs(tr - r), abs(tc - c))
                            if moving_piece.name == "Pawn" and new_k_dist > 3 and old_k_dist <= 3:
                                score_delta -= 15.0 # Keep defensive wall intact
                        elif p.color == self.enemy_color:
                            old_dist = abs(fr - r) + abs(fc - c)
                            new_dist = abs(tr - r) + abs(tc - c)
                            if new_dist < old_dist:
                                score_delta += (40 - new_dist) * 2.0

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None
        best_score = -float('inf')
        best_choices = []
        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)
        return random.choice(best_choices) if best_choices else random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #e2a100;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #e2a100;">
            ⚔️ DOUBLE-LAYER UNION WAR MATRIX &nbsp;|&nbsp; Global Match Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Monarch Files: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Monarch Files: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #e2a100; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Tactical Deployment Logs:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""
            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize Game Matrix
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Dual frontline defense shields organized. Match live."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Global formation capture complete."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Global formation capture complete."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            promo_note = " -> ♕ PROMOTED" if moving_piece.name == "Pawn" and (to_sq[0] == 15 or to_sq[0] == 0) else ""
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""

            log_line = f"Action {step+1}: {moving_piece.color} {moving_piece.name} maneuvers to {to_sq}{promo_note}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED VECTOR ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.015)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.15)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None
        self.position_history = {"White": [], "Black": []}

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        # --- WHITE INITIALIZATION ---
        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[0][i] = ChessPiece(white_pieces_order[i], "White")
            board[1][i] = ChessPiece("Pawn", "White")
            board[2][i] = ChessPiece("Pawn", "White")

        # --- BLACK INITIALIZATION ---
        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")
            board[14][i] = ChessPiece("Pawn", "Black")
            board[13][i] = ChessPiece("Pawn", "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False, ignore_moved_restriction=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and not ignore_moved_restriction and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- ADVANCED DOUBLE-LAYER PAWN ENGINE ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            layer_1_start = 1 if piece.color == "White" else 14
            layer_2_start = 2 if piece.color == "White" else 13

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == layer_1_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))
                        if self.board[r + 3 * direction][c] is None:
                            moves.append((r + 3 * direction, c))
                elif r == layer_2_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))

            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        if piece.name == "Pawn" and (tr == 15 or tr == 0):
            piece = ChessPiece("Queen", piece.color)

        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.position_history[piece.color].append(to_sq)
        if len(self.position_history[piece.color]) > 6:
            self.position_history[piece.color].pop(0)

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- THE HIGH-IQ THREAT-AWARE AI ROBOT ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 45, "Rook": 65, "Queen": 120, "King": 9999}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0

        # 1. Capture Value Weighting
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 20

        # 2. Positional Progression Values
        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 2.5

        # Avoid basic repetitions
        if to_sq in game.position_history[self.color]:
            score_delta -= 15.0

        # 3. HIGH-INTELLECT THREAT DETECTION MATRIX
        # We simulate the threat profile on the grid *before* finalizing
        enemy_attacks_target = False
        friendly_guards_target = False

        # Scan everything looking at the target square
        for r in range(game.rows):
            for c in range(game.cols):
                # Don't look from the piece that is currently moving away
                if (r, c) == (fr, fc): continue

                p = game.board[r][c]
                if p:
                    # Is an enemy piece threatening our landing spot?
                    if p.color == self.enemy_color:
                        # Quick check: if the target square is in their attack vector array
                        valid_enemy_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_enemy_moves:
                            enemy_attacks_target = True
                    # Is a friendly piece guarding our landing spot?
                    elif p.color == self.color:
                        valid_friendly_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_friendly_moves:
                            friendly_guards_target = True

        # ANTI-SUICIDE LOGIC GAUNTLET:
        if enemy_attacks_target:
            my_value = self.piece_values.get(moving_piece.name, 0)
            if not friendly_guards_target:
                # Completely unprotected suicide dive -> Apply massive penalty
                score_delta -= my_value * 50
            else:
                # Protected square, but if it's an expensive piece (like a Queen) trading for a cheap square, penalize it
                if moving_piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                    score_delta -= my_value * 15

        # 4. Phalanx Shield Formations (Pawn structures remain smart)
        if moving_piece.name == "Pawn":
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 6.0
            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 5.0

        # 5. Proximity Tracking (King Defense vs King Assassination)
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King":
                    if p.color == self.color:
                        old_k_dist = max(abs(fr - r), abs(fc - c))
                        new_k_dist = max(abs(tr - r), abs(tc - c))
                        if moving_piece.name == "Pawn" and new_k_dist > 3 and old_k_dist <= 3:
                            score_delta -= 25.0
                    elif p.color == self.enemy_color:
                        old_dist = abs(fr - r) + abs(fc - c)
                        new_dist = abs(tr - r) + abs(tc - c)
                        if new_dist < old_dist:
                            score_delta += (40 - new_dist) * 2.5

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_choices = []

        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)

        if best_choices:
            return random.choice(best_choices)
        return random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing Matrix..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #ff4a4a;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #ff4a4a;">
            🧠 HIGH-IQ THREAT DEFLECTION INTELLIGENCE GRID &nbsp;|&nbsp; Global Match Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Monarch Files: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Monarch Files: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #ff4a4a; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Threat Assessment Stream Logs:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""
            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize Game Matrix
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Threat analyzer matrices running calibration vectors."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Threat intelligence domination victory."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Threat intelligence domination victory."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            promo_note = " -> ♕ PROMOTED" if moving_piece.name == "Pawn" and (to_sq[0] == 15 or to_sq[0] == 0) else ""
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""

            log_line = f"Action {step+1}: Verified {moving_piece.color} {moving_piece.name} safely to {to_sq}{promo_note}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED VECTOR ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.012)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.12)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None
        self.position_history = {"White": [], "Black": []}

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        # --- WHITE INITIALIZATION ---
        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[0][i] = ChessPiece(white_pieces_order[i], "White")
            board[1][i] = ChessPiece("Pawn", "White")
            board[2][i] = ChessPiece("Pawn", "White")

        # --- BLACK INITIALIZATION ---
        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")
            board[14][i] = ChessPiece("Pawn", "Black")
            board[13][i] = ChessPiece("Pawn", "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False, ignore_moved_restriction=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and not ignore_moved_restriction and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- ADVANCED DOUBLE-LAYER PAWN ENGINE ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            layer_1_start = 1 if piece.color == "White" else 14
            layer_2_start = 2 if piece.color == "White" else 13

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == layer_1_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))
                        if self.board[r + 3 * direction][c] is None:
                            moves.append((r + 3 * direction, c))
                elif r == layer_2_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))

            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        if piece.name == "Pawn" and (tr == 15 or tr == 0):
            piece = ChessPiece("Queen", piece.color)

        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.position_history[piece.color].append(to_sq)
        if len(self.position_history[piece.color]) > 6:
            self.position_history[piece.color].pop(0)

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- THE HIGH-IQ THREAT-AWARE AI ROBOT ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 45, "Rook": 65, "Queen": 120, "King": 9999}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0

        # 1. Capture Value Weighting
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 20

        # 2. Positional Progression Values
        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 2.5

        # Avoid basic repetitions
        if to_sq in game.position_history[self.color]:
            score_delta -= 15.0

        # 3. HIGH-INTELLECT THREAT DETECTION MATRIX
        # We simulate the threat profile on the grid *before* finalizing
        enemy_attacks_target = False
        friendly_guards_target = False

        # Scan everything looking at the target square
        for r in range(game.rows):
            for c in range(game.cols):
                # Don't look from the piece that is currently moving away
                if (r, c) == (fr, fc): continue

                p = game.board[r][c]
                if p:
                    # Is an enemy piece threatening our landing spot?
                    if p.color == self.enemy_color:
                        # Quick check: if the target square is in their attack vector array
                        valid_enemy_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_enemy_moves:
                            enemy_attacks_target = True
                    # Is a friendly piece guarding our landing spot?
                    elif p.color == self.color:
                        valid_friendly_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_friendly_moves:
                            friendly_guards_target = True

        # ANTI-SUICIDE LOGIC GAUNTLET:
        if enemy_attacks_target:
            my_value = self.piece_values.get(moving_piece.name, 0)
            if not friendly_guards_target:
                # Completely unprotected suicide dive -> Apply massive penalty
                score_delta -= my_value * 50
            else:
                # Protected square, but if it's an expensive piece (like a Queen) trading for a cheap square, penalize it
                if moving_piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                    score_delta -= my_value * 15

        # 4. Phalanx Shield Formations (Pawn structures remain smart)
        if moving_piece.name == "Pawn":
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 6.0
            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 5.0

        # 5. Proximity Tracking (King Defense vs King Assassination)
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King":
                    if p.color == self.color:
                        old_k_dist = max(abs(fr - r), abs(fc - c))
                        new_k_dist = max(abs(tr - r), abs(tc - c))
                        if moving_piece.name == "Pawn" and new_k_dist > 3 and old_k_dist <= 3:
                            score_delta -= 25.0
                    elif p.color == self.enemy_color:
                        old_dist = abs(fr - r) + abs(fc - c)
                        new_dist = abs(tr - r) + abs(tc - c)
                        if new_dist < old_dist:
                            score_delta += (40 - new_dist) * 2.5

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_choices = []

        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)

        if best_choices:
            return random.choice(best_choices)
        return random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing Matrix..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #ff4a4a;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #ff4a4a;">
            🧠 HIGH-IQ THREAT DEFLECTION INTELLIGENCE GRID &nbsp;|&nbsp; Global Match Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Monarch Files: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Monarch Files: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #ff4a4a; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Threat Assessment Stream Logs:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""
            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize Game Matrix
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Threat analyzer matrices running calibration vectors."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Threat intelligence domination victory."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Threat intelligence domination victory."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            promo_note = " -> ♕ PROMOTED" if moving_piece.name == "Pawn" and (to_sq[0] == 15 or to_sq[0] == 0) else ""
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""

            log_line = f"Action {step+1}: Verified {moving_piece.color} {moving_piece.name} safely to {to_sq}{promo_note}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED VECTOR ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.012)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.12)

In [ ]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None
        self.position_history = {"White": [], "Black": []}

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        # --- WHITE INITIALIZATION ---
        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[0][i] = ChessPiece(white_pieces_order[i], "White")
            board[1][i] = ChessPiece("Pawn", "White")
            board[2][i] = ChessPiece("Pawn", "White")

        # --- BLACK INITIALIZATION ---
        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")
            board[14][i] = ChessPiece("Pawn", "Black")
            board[13][i] = ChessPiece("Pawn", "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False, ignore_moved_restriction=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and not ignore_moved_restriction and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- ADVANCED DOUBLE-LAYER PAWN ENGINE ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            layer_1_start = 1 if piece.color == "White" else 14
            layer_2_start = 2 if piece.color == "White" else 13

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == layer_1_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))
                        if self.board[r + 3 * direction][c] is None:
                            moves.append((r + 3 * direction, c))
                elif r == layer_2_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))

            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        if piece.name == "Pawn" and (tr == 15 or tr == 0):
            piece = ChessPiece("Queen", piece.color)

        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.position_history[piece.color].append(to_sq)
        if len(self.position_history[piece.color]) > 6:
            self.position_history[piece.color].pop(0)

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 45, "Rook": 65, "Queen": 120, "King": 9999}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 20

        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 2.5

        if to_sq in game.position_history[self.color]:
            score_delta -= 15.0

        enemy_attacks_target = False
        friendly_guards_target = False

        for r in range(game.rows):
            for c in range(game.cols):
                if (r, c) == (fr, fc): continue
                p = game.board[r][c]
                if p:
                    if p.color == self.enemy_color:
                        valid_enemy_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_enemy_moves:
                            enemy_attacks_target = True
                    elif p.color == self.color:
                        valid_friendly_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_friendly_moves:
                            friendly_guards_target = True

        if enemy_attacks_target:
            my_value = self.piece_values.get(moving_piece.name, 0)
            if not friendly_guards_target:
                score_delta -= my_value * 50
            else:
                if moving_piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                    score_delta -= my_value * 15

        if moving_piece.name == "Pawn":
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 6.0
            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 5.0

        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King":
                    if p.color == self.color:
                        old_k_dist = max(abs(fr - r), abs(fc - c))
                        new_k_dist = max(abs(tr - r), abs(tc - c))
                        if moving_piece.name == "Pawn" and new_k_dist > 3 and old_k_dist <= 3:
                            score_delta -= 25.0
                    elif p.color == self.enemy_color:
                        old_dist = abs(fr - r) + abs(fc - c)
                        new_dist = abs(tr - r) + abs(tc - c)
                        if new_dist < old_dist:
                            score_delta += (40 - new_dist) * 2.5

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None
        best_score = -float('inf')
        best_choices = []
        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)
        return random.choice(best_choices) if best_choices else random.choice(legal_moves)

# --- HTML RENDERING ENGINE (HEIGHT SET TO 90px FOR 4 TOTAL LINES) ---
def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing Matrix..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #ff4a4a;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #ff4a4a;">
            🧠 HIGH-IQ THREAT DEFLECTION INTELLIGENCE GRID &nbsp;|&nbsp; Global Match Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Monarch Files: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Monarch Files: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #ff4a4a; font-size: 12px; color: #ddd; line-height: 1.4; height: 90px; overflow-y: auto; box-sizing: border-box;">
            <b>Threat Assessment Stream Logs:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""
            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [ ]:
# Initialize Game Matrix
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Threat analyzer matrices running calibration vectors."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Threat intelligence domination victory."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Threat intelligence domination victory."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            promo_note = " -> ♕ PROMOTED" if moving_piece.name == "Pawn" and (to_sq[0] == 15 or to_sq[0] == 0) else ""
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""

            log_line = f"Action {step+1}: Verified {moving_piece.color} {moving_piece.name} safely to {to_sq}{promo_note}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED VECTOR ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.012)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.12)